**Assignment: SQL Notebook for Peer Assignment**

In [2]:
import pandas as pd
import csv, sqlite3
import prettytable
prettytable.DEFAULT="DEFAULT"

In [3]:
con=sqlite3.connect("my_data1.db")
cur=con.cursor()

df=pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL",con, if_exists="replace",index=False, method="multi")

101

In [5]:
%load_ext sql

In [6]:
%sql sqlite:///my_data1.db

In [7]:
%%sql

DROP TABLE IF EXISTS SPACEXTABLE;

CREATE TABLE SPACEXTABLE as SELECT * FROM SPACEXTBL WHERE Date is not null


 * sqlite:///my_data1.db
Done.
Done.


[]

In [9]:
%%sql
PRAGMA table_info("SPACEXTBL")

 * sqlite:///my_data1.db
Done.


cid,name,type,notnull,dflt_value,pk
0,Date,TEXT,0,None,0
1,Time (UTC),TEXT,0,None,0
2,Booster_Version,TEXT,0,None,0
3,Launch_Site,TEXT,0,None,0
4,Payload,TEXT,0,None,0
5,PAYLOAD_MASS__KG_,INTEGER,0,None,0
6,Orbit,TEXT,0,None,0
7,Customer,TEXT,0,None,0
8,Mission_Outcome,TEXT,0,None,0
9,Landing_Outcome,TEXT,0,None,0


In [10]:
%%sql
SELECT DISTINCT "Launch_Site" from SPACEXTBL;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


In [11]:
%%sql
SELECT * FROM SPACEXTBL WHERE "Launch_Site" LIKE 'CCA%' LIMIT 5

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [13]:
%%sql
SELECT SUM("PAYLOAD_MASS__KG_") as total 
from "SPACEXTBL"
WHERE "Customer" LIKE 'NASA (CRS)%'

 * sqlite:///my_data1.db
Done.


total
48213


In [14]:
%%sql
SELECT AVG("PAYLOAD_MASS__KG_") as average_mass 
from "SPACEXTBL" 
WHERE "Booster_Version" LIKE 'F9 v1.1';

 * sqlite:///my_data1.db
Done.


average_mass
2928.4


In [19]:
%sql SELECT DISTINCT "Landing_Outcome" FROM "SPACEXTBL"

 * sqlite:///my_data1.db
Done.


Landing_Outcome
Failure (parachute)
No attempt
Uncontrolled (ocean)
Controlled (ocean)
Failure (drone ship)
Precluded (drone ship)
Success (ground pad)
Success (drone ship)
Success
Failure


In [18]:
%%sql
SELECT MIN("Date") FROM "SPACEXTBL" 
WHERE "Landing_Outcome" = 'Success (ground pad)';

 * sqlite:///my_data1.db
Done.


"MIN(""Date"")"
2015-12-22


In [20]:
%%sql
SELECT "Booster_Version" FROM "SPACEXTBL"
WHERE "Landing_Outcome"='Success (drone ship)' AND "PAYLOAD_MASS__KG_" BETWEEN 4000 AND 6000;

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


In [21]:
%%sql

SELECT "Booster_Version"
FROM "SPACEXTBL"
WHERE "Landing_Outcome" = 'Success (drone ship)'
AND "PAYLOAD_MASS__KG_" > 4000
AND "PAYLOAD_MASS__KG_" < 6000;

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


In [ ]:
%%sql
SELECT "Landing_Outcome", COUNT() as "totales" FROM "SPACEXTBL" WHERE "Landing_Outcome" 
LIKE 'Failure%' OR "Landing_Outcome" LIKE 'Success%'
GROUP BY ("Landing_Outcome") 

 * sqlite:///my_data1.db
Done.


Landing_Outcome,totales
Failure,3
Failure (drone ship),5
Failure (parachute),2
Success,38
Success (drone ship),14
Success (ground pad),9


In [30]:
%%sql

SELECT
    CASE
        WHEN "Landing_Outcome" LIKE 'Success%' THEN 'Success'
        WHEN "Landing_Outcome" LIKE 'Failure%' THEN 'Failure'
    END AS Outcome,
    COUNT(*) AS Totales
FROM "SPACEXTBL"
WHERE "Landing_Outcome" LIKE 'Success%'
   OR "Landing_Outcome" LIKE 'Failure%'
GROUP BY
    CASE
        WHEN "Landing_Outcome" LIKE 'Success%' THEN 'Success'
        WHEN "Landing_Outcome" LIKE 'Failure%' THEN 'Failure'
    END;

 * sqlite:///my_data1.db
Done.


Outcome,Totales
Failure,10
Success,61


In [31]:
%%sql
SELECT "Booster_Version", "PAYLOAD_MASS__KG_" FROM "SPACEXTBL"
WHERE "PAYLOAD_MASS__KG_" = (
    SELECT MAX("PAYLOAD_MASS__KG_") from "SPACEXTBL"
)

 * sqlite:///my_data1.db
Done.


Booster_Version,PAYLOAD_MASS__KG_
F9 B5 B1048.4,15600
F9 B5 B1049.4,15600
F9 B5 B1051.3,15600
F9 B5 B1056.4,15600
F9 B5 B1048.5,15600
F9 B5 B1051.4,15600
F9 B5 B1049.5,15600
F9 B5 B1060.2,15600
F9 B5 B1058.3,15600
F9 B5 B1051.6,15600


In [34]:
%%sql
SELECT 
substr("Date",6,2) as Month, 
substr("Date",1,4) as Year, "Landing_Outcome","Booster_Version", "Launch_Site" FROM "SPACEXTBL" 
WHERE substr("Date",1,4) ='2015' and "Landing_Outcome" LIKE "Failure%"

 * sqlite:///my_data1.db
Done.


Month,Year,Landing_Outcome,Booster_Version,Launch_Site
01,2015,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,2015,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


In [38]:
%%sql
SELECT Landing_Outcome, COUNT() from "SPACEXTBL" 
WHERE Date BETWEEN '2010-06-04' and '2017-03-20' 
GROUP BY Landing_Outcome
ORDER BY COUNT() DESC;

 * sqlite:///my_data1.db
Done.


Landing_Outcome,COUNT()
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
